# Visualizar Código Fonte de Função Oracle

Este notebook consulta o código fonte de funções PL/SQL no Oracle através do dicionário de dados.

## ⚠️ IMPORTANTE: Ambiente de Execução

**Este notebook funciona melhor no Jupyter Lab (Docker):**

1. **Acesse o Jupyter Lab no navegador:**
   ```
   http://localhost:8888/lab
   ```

2. **No Jupyter Lab, o Spark estará disponível automaticamente**

3. **Se executar localmente (VS Code/Cursor):**
   - PySpark pode não estar disponível
   - Pode precisar de Oracle Instant Client para criptografia nativa
   - Recomendado: use o Jupyter Lab via navegador

In [1]:
import sys
import os

project_root = os.path.dirname(os.path.dirname(os.path.abspath('')))
is_docker = os.path.exists('/app')
if is_docker:
    sys.path.insert(0, '/app')
    print("✓ Ambiente Docker detectado")
else:
    sys.path.insert(0, project_root)
    print(f"⚠ Ambiente local detectado: {project_root}")
    print("💡 Dica: Para melhor compatibilidade, execute no Jupyter Lab:")
    print("   http://localhost:8888/lab")

try:
    from config.settings import oracle_config
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(project_root, '.env'))
    
    class OracleConfig:
        def __init__(self):
            self.host = os.getenv('ORACLE_HOST', 'localhost')
            self.port = int(os.getenv('ORACLE_PORT', '1521'))
            self.service = os.getenv('ORACLE_SERVICE', '')
            self.user = os.getenv('ORACLE_USER', '')
            self.password = os.getenv('ORACLE_PASSWORD', '')
    
    oracle_config = OracleConfig()

use_spark = False
try:
    from pyspark.sql import SparkSession
    use_spark = True
    print("✓ Modo Spark disponível (recomendado)")
except ImportError as e1:
    if not is_docker:
        pyenv_site_packages = os.path.expanduser("~/.pyenv/versions/3.11.14/lib/python3.11/site-packages")
        if os.path.exists(pyenv_site_packages):
            if pyenv_site_packages not in sys.path:
                sys.path.insert(0, pyenv_site_packages)
            pyspark_path = os.path.join(pyenv_site_packages, "pyspark")
            if os.path.exists(pyspark_path):
                try:
                    from pyspark.sql import SparkSession
                    use_spark = True
                    print("✓ PySpark encontrado no pyenv global")
                except ImportError as e2:
                    error_msg = str(e2)
                    if "_with_origin" in error_msg or "pyspark.errors.utils" in error_msg:
                        print("⚠ PySpark no pyenv mas incompatível com Python 3.11")
                        print("   Erro: incompatibilidade de versão do PySpark")
                        print("\n💡 SOLUÇÃO RECOMENDADA:")
                        print("   Execute no Jupyter Lab (PySpark configurado corretamente):")
                        print("   http://localhost:8888/lab")
                        print("\n   Ou atualize PySpark no pyenv:")
                        print("   pip install --upgrade pyspark")
                    else:
                        print(f"⚠ PySpark no pyenv mas erro ao importar: {error_msg[:100]}")
                    use_spark = False
            else:
                print(f"⚠ PySpark não encontrado em: {pyspark_path}")
        else:
            print(f"⚠ Caminho pyenv não encontrado: {pyenv_site_packages}")
    
    if not use_spark:
        print("⚠ PySpark não disponível")
        if not is_docker:
            print("💡 Execute no Jupyter Lab para ter acesso ao Spark:")
            print("   http://localhost:8888/lab")
        print("   Tentando usar oracledb diretamente...")
        try:
            import oracledb
            try:
                oracledb.init_oracle_client()
                print("✓ Modo thick ativado (Oracle Client)")
            except Exception as e:
                if "DPI-1047" in str(e) or "Cannot locate" in str(e):
                    print("⚠ Modo thin (sem Oracle Client)")
                    print("   Oracle pode exigir criptografia nativa")
                else:
                    print(f"⚠ Aviso ao inicializar cliente: {str(e)[:50]}")
        except ImportError:
            print("✗ Erro: oracledb não encontrado. Instale com: pip install oracledb")

⚠ Ambiente local detectado: /Users/amaro/Documents/BeAnaityc
💡 Dica: Para melhor compatibilidade, execute no Jupyter Lab:
   http://localhost:8888/lab
✓ PySpark encontrado no pyenv global


In [2]:
if use_spark:
    try:
        spark_builder = SparkSession.builder.appName("ViewOracleFunction")
        
        if not is_docker:
            spark_builder = spark_builder.config(
                "spark.jars.packages",
                "com.oracle.database.jdbc:ojdbc8:23.2.0.0"
            )
        
        spark = spark_builder.getOrCreate()
        oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_config.host}:{oracle_config.port}/{oracle_config.service}"
        print(f"✓ Spark Session criada")
        if not is_docker:
            print("✓ Driver Oracle JDBC configurado via Maven")
        print(f"✓ JDBC URL: {oracle_jdbc_url}")
    except Exception as e:
        error_msg = str(e)
        if "ClassNotFoundException" in error_msg or "oracle.jdbc" in error_msg:
            print("✗ Erro: Driver Oracle JDBC não encontrado")
            print("\n💡 SOLUÇÃO RECOMENDADA:")
            print("   Execute no Jupyter Lab (driver já configurado):")
            print("   http://localhost:8888/lab")
            print("\n   Ou baixe o driver manualmente:")
            print("   1. Baixe: https://www.oracle.com/database/technologies/appdev/jdbc-downloads.html")
            print("   2. Configure: spark.config('spark.jars', '/caminho/para/ojdbc8.jar')")
        else:
            print(f"⚠ Erro ao criar Spark Session: {error_msg[:200]}")
            print("💡 Certifique-se de que o Spark Master está rodando:")
            print("   docker compose up -d spark-master spark-worker-1")
        use_spark = False
        print(f"\n✓ Configuração Oracle: {oracle_config.host}:{oracle_config.port}/{oracle_config.service}")
else:
    print(f"✓ Configuração Oracle: {oracle_config.host}:{oracle_config.port}/{oracle_config.service}")
    if not is_docker:
        print("\n💡 Para usar Spark (recomendado):")
        print("   1. Acesse: http://localhost:8888/lab")
        print("   2. Execute este notebook no Jupyter Lab")
        print("   3. Ou inicie o Spark: docker compose up -d spark-master spark-worker-1")

26/01/26 13:00:41 WARN Utils: Your hostname, Joses-Mac-Studio.local resolves to a loopback address: 127.0.0.1; using 192.168.2.32 instead (on interface en1)
26/01/26 13:00:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/amaro/.ivy2/cache
The jars for the packages stored in: /Users/amaro/.ivy2/jars
com.oracle.database.jdbc#ojdbc8 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c22b869-ebd2-428c-aae4-518e8a9621b0;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/amaro/.pyenv/versions/3.11.14/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.oracle.database.jdbc#ojdbc8;23.2.0.0 in central
downloading https://repo1.maven.org/maven2/com/oracle/database/jdbc/ojdbc8/23.2.0.0/ojdbc8-23.2.0.0.jar ...
	[SUCCESSFUL ] com.oracle.database.jdbc#ojdbc8;23.2.0.0!ojdbc8.jar (632ms)
:: resolution report :: resolve 724ms :: artifacts dl 633ms
	:: modules in use:
	com.oracle.database.jdbc#ojdbc8;23.2.0.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   1   |   1   |   0   ||   1   |   1   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-6c22b869-ebd2-428c-aae4-518e8a9621b0
	confs: [default]
	1 artifacts copied, 0 already retrieved (6684kB/7ms)
26/01/26 13:00:42 WAR

✓ Spark Session criada
✓ Driver Oracle JDBC configurado via Maven
✓ JDBC URL: jdbc:oracle:thin:@//localhost:1521/bi.grupotracker.com.br


In [3]:
schema = "BISTAGE"
function_name = "F_CONSULTA_INST_BI"

if use_spark:
    query = f"""
    (SELECT line, text
    FROM all_source
    WHERE owner = UPPER('{schema}')
    AND name = UPPER('{function_name}')
    AND type = 'FUNCTION'
    ORDER BY line)
    """
    
    df = (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("query", query)
        .option("user", oracle_config.user)
        .option("password", oracle_config.password)
        .load()
    )
    
    rows = df.collect()
    
    if rows:
        print("=" * 80)
        print(f"CÓDIGO FONTE: {schema}.{function_name}")
        print("=" * 80)
        print()
        
        for row in rows:
            line_num = row['LINE']
            text = row['TEXT']
            print(f"{line_num:4d} | {text.rstrip()}")
        
        print()
        print("=" * 80)
    else:
        print(f"Função {schema}.{function_name} não encontrada.")
else:
    try:
        import oracledb
        
        dsn = oracledb.makedsn(
            host=oracle_config.host,
            port=oracle_config.port,
            service_name=oracle_config.service
        )
        
        conn = oracledb.connect(
            user=oracle_config.user,
            password=oracle_config.password,
            dsn=dsn
        )
        
        cursor = conn.cursor()
        
        query = """
        SELECT line, text
        FROM all_source
        WHERE owner = UPPER(:schema)
        AND name = UPPER(:function_name)
        AND type = 'FUNCTION'
        ORDER BY line
        """
        
        cursor.execute(query, schema=schema, function_name=function_name)
        rows = cursor.fetchall()
        
        if rows:
            print("=" * 80)
            print(f"CÓDIGO FONTE: {schema}.{function_name}")
            print("=" * 80)
            print()
            
            for line_num, text in rows:
                print(f"{line_num:4d} | {text.rstrip()}")
            
            print()
            print("=" * 80)
        else:
            print(f"Função {schema}.{function_name} não encontrada.")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        error_msg = str(e)
        if "DPY-3001" in error_msg or "thick mode" in error_msg.lower():
            print("✗ Erro: Oracle exige criptografia nativa (modo thick)")
            print("\nSolução: Instale Oracle Instant Client ou use Docker:")
            print("  docker compose up -d jupyter")
        else:
            print(f"✗ Erro: {error_msg}")

Função BISTAGE.F_CONSULTA_INST_BI não encontrada.


In [4]:
if use_spark:
    query_list = f"""
    (SELECT name, type
    FROM all_source
    WHERE owner = UPPER('{schema}')
    AND type IN ('FUNCTION', 'PROCEDURE')
    GROUP BY name, type
    ORDER BY name)
    """
    
    df_list = (
        spark.read.format("jdbc")
        .option("url", oracle_jdbc_url)
        .option("driver", "oracle.jdbc.OracleDriver")
        .option("query", query_list)
        .option("user", oracle_config.user)
        .option("password", oracle_config.password)
        .load()
    )
    
    print(f"Funções e Procedures no schema {schema}:")
    df_list.show(truncate=False)
else:
    try:
        import oracledb
        
        dsn = oracledb.makedsn(
            host=oracle_config.host,
            port=oracle_config.port,
            service_name=oracle_config.service
        )
        
        conn = oracledb.connect(
            user=oracle_config.user,
            password=oracle_config.password,
            dsn=dsn
        )
        
        cursor = conn.cursor()
        
        cursor.execute("""
            SELECT DISTINCT name, type
            FROM all_source
            WHERE owner = UPPER(:schema)
            AND type IN ('FUNCTION', 'PROCEDURE')
            ORDER BY name
        """, schema=schema)
        
        rows = cursor.fetchall()
        
        if rows:
            print(f"Funções e Procedures no schema {schema}:")
            print("-" * 80)
            for name, obj_type in rows:
                print(f"  {name:50} ({obj_type})")
        else:
            print(f"Nenhuma função/procedure encontrada no schema {schema}")
        
        cursor.close()
        conn.close()
        
    except Exception as e:
        print(f"✗ Erro ao listar funções: {str(e)}")

Funções e Procedures no schema BISTAGE:


+----+----+
|NAME|TYPE|
+----+----+
+----+----+



In [5]:
if use_spark:
    spark.stop()
    print("✓ Spark Session finalizada")

✓ Spark Session finalizada
